# Persistent entropy using signed correlation distance

This notebook reproduces the persistent-entropy analysis in which the distance matrix is defined as `1 - correlation`. It is retained as a separate analysis because it corresponds to a distinct methodological choice from the absolute-correlation version.


## Data loading


In [1]:
# Private matrix-data configuration and paired loading
from pathlib import Path
import re
import glob
import numpy as np
import pandas as pd
import scipy
from scipy import stats
import matplotlib.pyplot as plt

PROJECT_DIR = Path.cwd()
CANDIDATE_DATA_DIRS = [
    PROJECT_DIR / "data_private" / "connectivity_matrices",
    PROJECT_DIR / "public_repository" / "data_private" / "connectivity_matrices",
    PROJECT_DIR,
    PROJECT_DIR.parent,
]

# In the public repository, place private .txt matrices under:
#   data_private\connectivity_matrices\
# The additional candidates support the local preparation workspace only.
DATA_DIR = next((path for path in CANDIDATE_DATA_DIRS if any(path.glob("*before*.txt"))), CANDIDATE_DATA_DIRS[0])

BEFORE_PATTERN = "*before*.txt"
AFTER_PATTERN = "*after*.txt"


def _subject_id(path):
    name = Path(path).name.lower()
    match = re.search(r"suj\s*[_-]?(\d+)|s[_-]?(\d+)", name)
    if match:
        return int(match.group(1) or match.group(2))
    return name


def _load_paired_connectivity_matrices(data_dir):
    before_files = sorted(Path(data_dir).glob(BEFORE_PATTERN), key=_subject_id)
    after_files = sorted(Path(data_dir).glob(AFTER_PATTERN), key=_subject_id)

    if len(before_files) == 0 or len(after_files) == 0:
        raise FileNotFoundError(
            f"No paired .txt connectivity matrices were found in {Path(data_dir).resolve()}. "
            "Set DATA_DIR to the private matrix folder."
        )

    before_ids = [_subject_id(path) for path in before_files]
    after_ids = [_subject_id(path) for path in after_files]
    if before_ids != after_ids:
        raise ValueError(
            "Before/after subject identifiers do not match.\n"
            f"before={before_ids}\nafter={after_ids}"
        )

    before_array = np.array([np.loadtxt(path) for path in before_files])
    after_array = np.array([np.loadtxt(path) for path in after_files])

    if before_array.shape != after_array.shape:
        raise ValueError(f"Shape mismatch: before={before_array.shape}, after={after_array.shape}")

    return before_array, after_array, before_files, after_files, before_ids


before, after, txt_files_before, txt_files_after, subject_ids = _load_paired_connectivity_matrices(DATA_DIR)

print(f"Data directory: {Path(DATA_DIR).resolve()}")
print(f"Paired subjects: {subject_ids}")
print(f"Number of before files: {len(txt_files_before)}")
print(f"Number of after files : {len(txt_files_after)}")
print(f"before shape: {before.shape}")
print(f"after shape : {after.shape}")


Data directory: D:\OneDrive\Coding\article_repository
Paired subjects: [1, 2, 3, 4, 5, 6, 7, 8, 9]
Number of before files: 9
Number of after files : 9
before shape: (9, 104, 104)
after shape : (9, 104, 104)


## Persistent entropy and paired statistics


In [2]:
# Cálculo entropia persistente

import gudhi
import scipy
import numpy as np
import matplotlib.pyplot as plt
from gudhi import representations
from tqdm import tqdm  # Importação da biblioteca de barra de progresso

# Alteração: Adicionado o parametro 'dim'
def c_pers(matrix, dim):
    dist_matrix = 1 - matrix
    
    # Obs: O .tolist() é importante para garantir compatibilidade, caso matrix seja numpy
    if isinstance(dist_matrix, np.ndarray):
        dist_matrix_list = dist_matrix.tolist()
    else:
        dist_matrix_list = dist_matrix

    scomplex = gudhi.RipsComplex(distance_matrix=dist_matrix_list, max_edge_length=1.99)

    # Alteração: Aumentado para 4 para permitir o cálculo da dimensão 3
    s_tree = scomplex.create_simplex_tree(max_dimension=4)

    a = s_tree.persistence(homology_coeff_field=2, min_persistence=0)
    
    # Alteração: Usa a dimensão passada como argumento
    b = s_tree.persistence_intervals_in_dimension(dim)
    
    gudhi.plot_persistence_barcode(a, legend=True)
    
    # Alteração: Desabilitado show e adicionado close para não acumular memória
    # plt.show()
    plt.close()

    # Lógica original mantida
    soma = 0
    for i in range(0, len(b)):
        if b[i][1] < float('inf'):
            soma += b[i][1]

    entropy = 0
    # Adicionei verificação simples para evitar divisão por zero se a soma for 0
    if soma > 0:
        for i in range(0, len(b)):
            if b[i][1] < float('inf'):        
                entropy += -(b[i][1]/soma)*np.log(b[i][1]/soma)
    
    return entropy

antes = []
depois = []

# Verificação se as listas existem e têm o mesmo tamanho
if 'before' in locals() and 'after' in locals() and len(before) == len(after):
    
    # --- ALTERAÇÃO AQUI: tqdm envolve o range para criar a barra ---
    # desc="Processando" adiciona um texto antes da barra
    for i in tqdm(range(0, len(before)), desc="Processando Sujeitos"):
        
        # Vetores temporários para este sujeito
        sujeito_antes_dims = []
        sujeito_depois_dims = []
        
        # Loop externo solicitando as dimensões 0, 1, 2 e 3
        for d in range(4):
            sujeito_antes_dims.append(c_pers(before[i], d))
            sujeito_depois_dims.append(c_pers(after[i], d))
        
        # Adiciona o vetor [d0, d1, d2, d3] à lista principal
        antes.append(sujeito_antes_dims)
        depois.append(sujeito_depois_dims)
else:
    print("Erro: Variáveis 'before'/'after' não definidas ou com tamanhos diferentes.")

print("\nProcessamento concluído.")
# print(antes) # Comentado pois pode ser muito grande para exibir
# print(depois)

# Exemplo de como chamar a estatística para a dimensão 0, se necessário:
# c_estat([x[0] for x in antes], [y[0] for y in depois], 0)


C:\ProgramData\anaconda3\Lib\site-packages\gudhi\persistence_graphical_tools.py:129: UserWarning: usetex mode requires TeX.
  warnings.warn("usetex mode requires TeX.")
Processando Sujeitos: 100%|██████████| 9/9 [8:32:56<00:00, 3419.56s/it]  


Processamento concluído.


In [3]:
import scipy.stats
import numpy as np

# 1. Sua função c_estat original
def c_estat(before, after, dim):
    print('\n' + '='*40)
    print('DIMENSÃO ' + str(dim))
    print('='*40)
    
    print('--- Teste de Normalidade ---')
    # O try/except evita que o código pare se todos os números forem iguais (variância zero)
    try:
        print('Shapiro_before: ' + str(scipy.stats.shapiro(before)))
        print('Shapiro_after: ' + str(scipy.stats.shapiro(after)))
    except Exception as e:
        print(f"Erro no Shapiro: {e}")

    print('\n--- Testes paramétricos e não-paramétricos ---')
    print('Ranksum: ' + str(scipy.stats.ranksums(before, after)))
    
    # Wilcoxon pode falhar se a diferença for zero em todos os casos, adicionado proteção
    try:
        print('Wilcoxon: ' + str(scipy.stats.wilcoxon(before, after)))
    except ValueError:
        print("Wilcoxon: Não foi possível calcular (diferenças zero ou tamanho insuficiente).")
        
    print('Paired T-Test: ' + str(scipy.stats.ttest_rel(before, after)))

# 2. Loop para processar as 4 dimensões (0, 1, 2, 3)
# Considera que 'antes' e 'depois' são listas de listas: [[d0, d1, d2, d3], [d0, d1, d2, d3], ...]

dimensoes_para_analisar = 4

for dim in range(dimensoes_para_analisar):
    # List comprehension para extrair a coluna da dimensão atual
    # Pega o índice [dim] de cada sujeito na lista 'antes'
    dados_antes_dim = [sujeito[dim] for sujeito in antes]
    
    # Pega o índice [dim] de cada sujeito na lista 'depois'
    dados_depois_dim = [sujeito[dim] for sujeito in depois]
    
    # Chama a função estatística para essa dimensão específica
    c_estat(dados_antes_dim, dados_depois_dim, dim)



DIMENSÃO 0
--- Teste de Normalidade ---
Shapiro_before: ShapiroResult(statistic=np.float64(0.8010908164127064), pvalue=np.float64(0.021015044965600797))
Shapiro_after: ShapiroResult(statistic=np.float64(0.9423933134471689), pvalue=np.float64(0.6073062511669656))

--- Testes paramétricos e não-paramétricos ---
Ranksum: RanksumsResult(statistic=np.float64(-0.7505683356701914), pvalue=np.float64(0.45291248342491686))
Wilcoxon: WilcoxonResult(statistic=np.float64(17.0), pvalue=np.float64(0.5703125))
Paired T-Test: TtestResult(statistic=np.float64(-0.3817785479035964), pvalue=np.float64(0.7125666573550467), df=np.int64(8))

DIMENSÃO 1
--- Teste de Normalidade ---
Shapiro_before: ShapiroResult(statistic=np.float64(0.9776902952388148), pvalue=np.float64(0.9513521796888332))
Shapiro_after: ShapiroResult(statistic=np.float64(0.9723358254193213), pvalue=np.float64(0.9139346836252811))

--- Testes paramétricos e não-paramétricos ---
Ranksum: RanksumsResult(statistic=np.float64(-0.044151078568834

In [4]:
import pingouin as pg
import pandas as pd

# Garante que o Pandas mostre todas as colunas no print (opcional, mas ajuda a ver o resultado)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print("=== ANÁLISE ESTATÍSTICA POR DIMENSÃO (WILCOXON) ===\n")

# Loop para as 4 dimensões (0, 1, 2, 3)
for dim in range(4):
    print(f"{'='*20} DIMENSÃO {dim} {'='*20}")
    
    # 1. Preparação dos dados: Extrai a coluna da dimensão atual de todos os sujeitos
    # Transformamos a lista de listas em dois vetores simples para aquela dimensão
    try:
        dados_antes_dim = [sujeito[dim] for sujeito in antes]
        dados_depois_dim = [sujeito[dim] for sujeito in depois]
        
        # 2. Execução do teste usando Pingouin
        # alternative='two-sided' é o padrão (bilateral)
        res = pg.wilcoxon(dados_antes_dim, dados_depois_dim)
        
        # 3. Exibição dos resultados
        print(res)
        print("\n") # Pula uma linha para organizar
        
    except IndexError:
        print(f"Erro: Não foi possível acessar dados para a dimensão {dim}. Verifique se 'antes' e 'depois' foram calculados corretamente.")
    except Exception as e:
        print(f"Erro ao calcular estatística para dimensão {dim}: {e}")


=== ANÁLISE ESTATÍSTICA POR DIMENSÃO (WILCOXON) ===

==================== DIMENSÃO 0 ====================
          W_val alternative     p_val       RBC      CLES
Wilcoxon   17.0   two-sided  0.570312 -0.244444  0.395062


==================== DIMENSÃO 1 ====================
          W_val alternative     p_val       RBC      CLES
Wilcoxon   21.0   two-sided  0.910156 -0.066667  0.493827


==================== DIMENSÃO 2 ====================
          W_val alternative     p_val  RBC      CLES
Wilcoxon   18.0   two-sided  0.652344  0.2  0.580247


==================== DIMENSÃO 3 ====================
          W_val alternative     p_val       RBC      CLES
Wilcoxon   17.0   two-sided  0.570312  0.244444  0.580247




In [5]:
###### Código Gemini


In [6]:
import numpy as np
from scipy.stats import rankdata, wilcoxon

# Assumindo que 'antes' e 'depois' já estão carregados na memória
# com 9 sujeitos e 4 dimensões cada.

num_dimensoes = 4

for dim in range(num_dimensoes):
    print(f"--- Resultados para a Dimensão {dim} (S_{dim}) ---")
    
    # 1. Extração dos dados para a dimensão atual
    s_antes = np.array([sujeito[dim] for sujeito in antes])
    s_depois = np.array([sujeito[dim] for sujeito in depois])
    
    # 2. Cálculo das diferenças
    diferencas = s_antes - s_depois
    diff_validas = diferencas[diferencas != 0]
    
    # Prevenção caso todos os valores sejam exatamente iguais (empate total)
    if len(diff_validas) == 0:
        print("Todas as diferenças são zero. Não é possível calcular os postos.\n")
        continue
        
    # 3. Ranqueamento e separação de W+ e W-
    postos = rankdata(np.abs(diff_validas))
    w_mais = np.sum(postos[diff_validas > 0])
    w_menos = np.sum(postos[diff_validas < 0])
    
    # 4. Cálculo do Teste de Wilcoxon e Valor-p
    estatistica_w, p_valor = wilcoxon(s_antes, s_depois)
    
    # 5. Cálculo da Correlação Bisserial de Postos (RBC)
    soma_w = w_mais + w_menos
    rbc = (w_mais - w_menos) / soma_w if soma_w != 0 else 0.0
    
    # Impressão dos resultados
    print(f"Estatística W: {estatistica_w}")
    print(f"Valor-p: {p_valor:.3f}")
    print(f"Tamanho de Efeito (RBC): {rbc:.3f}\n")


--- Resultados para a Dimensão 0 (S_0) ---
Estatística W: 17.0
Valor-p: 0.570
Tamanho de Efeito (RBC): -0.244

--- Resultados para a Dimensão 1 (S_1) ---
Estatística W: 21.0
Valor-p: 0.910
Tamanho de Efeito (RBC): -0.067

--- Resultados para a Dimensão 2 (S_2) ---
Estatística W: 18.0
Valor-p: 0.652
Tamanho de Efeito (RBC): 0.200

--- Resultados para a Dimensão 3 (S_3) ---
Estatística W: 17.0
Valor-p: 0.570
Tamanho de Efeito (RBC): 0.244



In [7]:
from scipy.stats import bootstrap

# Usaremos as diferenças da Dimensão 2 que já calculamos
# diferencas_s2 = s2_antes - s2_depois

# Definimos uma função que calcula a média (ou mediana) das diferenças
def estatistica_diferenca(diferencas_amostra):
    return np.median(diferencas_amostra)

# Calculamos o IC não-paramétrico de 95% via Bootstrap
resultado_ic = bootstrap((diferencas,), estatistica_diferenca, confidence_level=0.95, method='BCa')

limite_inferior = resultado_ic.confidence_interval.low
limite_superior = resultado_ic.confidence_interval.high

print(f"Intervalo de Confiança (95%): [{limite_inferior:.3f}, {limite_superior:.3f}]")


Intervalo de Confiança (95%): [-0.676, 1.054]


In [8]:
from statsmodels.stats.multitest import multipletests

# Coloque aqui os 4 valores-p que calculamos (Dimensões 0, 1, 2 e 3)
p_valores_originais = [0.734, 0.820, 0.027, 0.129]

# Aplicamos a correção FDR (método 'fdr_bh' = Benjamini-Hochberg)
rejeitar_hipotese, p_valores_corrigidos, _, _ = multipletests(p_valores_originais, alpha=0.05, method='fdr_bh')

print(f"Valores-p originais: {p_valores_originais}")
print(f"Valores-p corrigidos (FDR): {p_valores_corrigidos}")
print(f"É significativo após correção?: {rejeitar_hipotese}")


Valores-p originais: [0.734, 0.82, 0.027, 0.129]
Valores-p corrigidos (FDR): [0.82  0.82  0.108 0.258]
É significativo após correção?: [False False False False]
